# 01 — Clean the rainfall data and complete Task 1

Run this notebook after cloud setup. It reads the eight manifest-listed annual CSVs, applies visible cleaning rules, and produces pre-test rainfall summaries. The calculations are shown here so the notebook can stand on its own during marking.

In [ ]:
from pathlib import Path
from functools import reduce
import json
import os

from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StringType, StructField, StructType

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = json.loads((PROJECT_ROOT / "config" / "experiment.json").read_text())
assert config["years"] == list(range(2017, 2025))
assert config["alignment_tolerance_seconds"] == 0
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")


## Load and inspect the eight annual files

The manifest selects the input files. Each annual DataFrame gets its source year before the rows are combined, so row counts and exclusions can be traced back to the file.

In [ ]:
manifest = json.loads((PROJECT_ROOT / "manifests" / "input_manifest.json").read_text())
files = sorted(manifest["files"], key=lambda row: row["year"])
assert [row["year"] for row in files] == list(range(2017, 2025))

column_names = [
    "date", "timestamp", "update_timestamp", "station_id", "station_name",
    "station_device_id", "location_longitude", "location_latitude",
    "reading_update_timestamp", "reading_value", "reading_type", "reading_unit",
]
raw_schema = StructType(
    [StructField(name, StringType(), True) for name in column_names]
    + [StructField("_corrupt_record", StringType(), True)]
)

annual_frames = []
for record in files:
    source = f"gs://{BUCKET}/{record['relative_path']}"
    annual = (
        spark.read.schema(raw_schema)
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .option("encoding", "UTF-8")
        .csv(source)
        .withColumn("_source_year", F.lit(record["year"]).cast("int"))
        .withColumn("_source_file", F.lit(record["relative_path"]))
    )
    annual_frames.append(annual)
raw = reduce(lambda left, right: left.unionByName(right), annual_frames).persist(StorageLevel.DISK_ONLY)
raw.groupBy("_source_year").count().orderBy("_source_year").show(8, truncate=False)
raw.groupBy("_source_year", "reading_type", "reading_unit").count().orderBy("_source_year", F.desc("count")).show(30, truncate=False)


In [ ]:
def add_parsed_columns(frame):
    parsed = frame
    for name in ("timestamp", "update_timestamp", "reading_update_timestamp"):
        parsed = parsed.withColumn(f"_{name}_parsed", F.to_timestamp(F.trim(F.col(name))))
    parsed = parsed.withColumn("_date_parsed", F.to_date(F.trim(F.col("date"))))
    for name in ("reading_value", "location_longitude", "location_latitude"):
        parsed = parsed.withColumn(f"_{name}_numeric", F.trim(F.col(name)).cast("double"))
    return parsed


def is_finite(column):
    return column.isNotNull() & ~F.isnan(column) & (F.abs(column) != float("inf"))


parsed = add_parsed_columns(raw)
raw_quality = parsed.groupBy("_source_year").agg(
    F.count("*").alias("raw_rows"),
    F.sum(F.col("_corrupt_record").isNotNull().cast("long")).alias("malformed_rows"),
    F.sum(F.col("station_id").isNull().cast("long")).alias("missing_station_rows"),
    F.sum(F.col("_timestamp_parsed").isNull().cast("long")).alias("invalid_timestamp_rows"),
    F.sum(F.col("_reading_value_numeric").isNull().cast("long")).alias("invalid_rainfall_rows"),
)
raw_quality.orderBy("_source_year").show(8, truncate=False)


## Clean, align and deduplicate

Rows are retained only when station, timestamp, rainfall, unit, type and coordinates are valid. Timestamps must already lie on the exact five-minute grid. Conflicting records at one station/time key are excluded; matching repeats are reduced deterministically.

In [ ]:
def flag_reason(frame, condition, reason):
    return frame.withColumn(
        "exclusion_reason",
        F.when(F.col("exclusion_reason").isNotNull(), F.col("exclusion_reason"))
        .when(F.coalesce(condition, F.lit(True)), F.lit(reason)),
    )


def prepare_observations(raw_rows, config):
    frame = add_parsed_columns(raw_rows).withColumn("exclusion_reason", F.lit(None).cast("string"))
    rules = [
        (F.col("_corrupt_record").isNotNull(), "csv_parser_error"),
        (F.col("station_id").isNull() | (F.trim("station_id") == ""), "missing_station"),
        (F.col("_timestamp_parsed").isNull(), "invalid_timestamp"),
        (F.col("_date_parsed").isNull() | (F.col("_date_parsed") != F.to_date("_timestamp_parsed")), "date_mismatch"),
        (F.year("_timestamp_parsed") != F.col("_source_year"), "year_mismatch"),
        (~is_finite(F.col("_reading_value_numeric")), "invalid_rainfall"),
        (F.col("_reading_value_numeric") < 0, "negative_rainfall"),
        (F.trim("reading_unit") != "mm", "invalid_unit"),
        (F.trim("reading_type") != config["reading_type"], "invalid_reading_type"),
        (~is_finite(F.col("_location_longitude_numeric")) | ~is_finite(F.col("_location_latitude_numeric")), "invalid_coordinates"),
        (~F.col("_location_longitude_numeric").between(-180, 180) | ~F.col("_location_latitude_numeric").between(-90, 90), "coordinates_out_of_range"),
    ]
    for condition, reason in rules:
        frame = flag_reason(frame, condition, reason)

    frame = frame.withColumn("epoch", F.col("_timestamp_parsed").cast("long"))
    frame = frame.withColumn("alignment_offset_seconds", F.pmod("epoch", F.lit(300)))
    alignment = frame.groupBy("_source_year", "alignment_offset_seconds").count()
    frame = flag_reason(frame, F.col("alignment_offset_seconds") != 0, "off_grid_timestamp")
    scale = config["rainfall_integer_scale"]
    frame = frame.withColumn("rain_units", F.round(F.col("_reading_value_numeric") * scale).cast("long"))
    frame = flag_reason(
        frame,
        F.abs(F.col("rain_units") / scale - F.col("_reading_value_numeric")) > 1e-10,
        "unsupported_rainfall_precision",
    )
    rejected = frame.where("exclusion_reason IS NOT NULL")
    candidates = frame.where("exclusion_reason IS NULL").withColumn("station_id", F.trim("station_id"))
    return candidates, rejected, alignment


In [ ]:
def resolve_duplicate_keys(candidates):
    expected = column_names + ["_source_year"]
    exact_order = Window.partitionBy(*expected).orderBy("_source_file")
    ranked = candidates.withColumn("_exact_rank", F.row_number().over(exact_order))
    exact_repeats = ranked.where("_exact_rank > 1").withColumn("exclusion_reason", F.lit("exact_duplicate"))
    unique = ranked.where("_exact_rank = 1")

    variants = unique.groupBy("station_id", "epoch").agg(
        F.countDistinct(F.struct(
            "rain_units", "station_name", "station_device_id",
            "_location_longitude_numeric", "_location_latitude_numeric",
        )).alias("_variants")
    )
    key_order = Window.partitionBy("station_id", "epoch").orderBy(
        "_update_timestamp_parsed", "_reading_update_timestamp_parsed", "_source_file"
    )
    ranked_keys = unique.join(variants, ["station_id", "epoch"]).withColumn(
        "_key_rank", F.row_number().over(key_order)
    )
    conflicts = ranked_keys.where("_variants > 1").withColumn("exclusion_reason", F.lit("conflicting_key"))
    repeats = ranked_keys.where("_variants = 1 AND _key_rank > 1").withColumn(
        "exclusion_reason", F.lit("same_measurement_duplicate")
    )
    retained = ranked_keys.where("_variants = 1 AND _key_rank = 1")
    rejected = exact_repeats.unionByName(conflicts, allowMissingColumns=True).unionByName(
        repeats, allowMissingColumns=True
    )
    return retained, rejected


def select_clean_columns(retained):
    return retained.select(
        "station_id", "epoch", "rain_units",
        F.col("_timestamp_parsed").alias("observation_time"),
        F.col("_reading_value_numeric").alias("rain_mm"),
        F.col("_location_longitude_numeric").alias("longitude"),
        F.col("_location_latitude_numeric").alias("latitude"),
        F.col("_source_year").alias("year"),
        F.to_date("_timestamp_parsed").alias("local_date"),
        F.month("_timestamp_parsed").alias("month"),
        F.hour("_timestamp_parsed").alias("hour"),
        F.col("_update_timestamp_parsed").alias("update_time"),
        F.col("_reading_update_timestamp_parsed").alias("reading_update_time"),
        "station_name", "station_device_id", "_source_file", "alignment_offset_seconds",
    )


In [ ]:
candidates, rejected_first_pass, alignment = prepare_observations(raw, config)
retained, rejected_duplicates = resolve_duplicate_keys(candidates)
clean = select_clean_columns(retained).persist(StorageLevel.DISK_ONLY)
rejected = rejected_first_pass.unionByName(rejected_duplicates, allowMissingColumns=True)
exclusions = rejected.groupBy("_source_year", "exclusion_reason").count()
quality = (
    raw.groupBy("_source_year").count().withColumnRenamed("count", "raw_rows")
    .join(clean.groupBy("year").agg(F.count("*").alias("retained_rows"), F.countDistinct("station_id").alias("stations")),
          F.col("_source_year") == F.col("year"), "left")
    .drop("year").withColumnRenamed("_source_year", "year").fillna(0)
)
clean.write.mode("overwrite").partitionBy("year").parquet(f"{ROOT}/cleaned")
rejected.write.mode("overwrite").partitionBy("_source_year").parquet(f"{ROOT}/rejected")
quality.write.mode("overwrite").json(f"{ROOT}/quality_by_year")
exclusions.write.mode("overwrite").json(f"{ROOT}/exclusion_counts")
alignment.write.mode("overwrite").json(f"{ROOT}/alignment_distribution")
quality.orderBy("year").show(8, truncate=False)
exclusions.orderBy("_source_year", "exclusion_reason").show(100, truncate=False)


## Coverage and missing intervals

Coverage uses the full calendar as its denominator, including absent station months. Gaps are measured between consecutive retained observations; no rainfall values are interpolated.

In [ ]:
months_observed = clean.groupBy("station_id", "year", "month").agg(
    F.count("*").alias("valid_slots"),
    F.countDistinct("local_date").alias("observed_days"),
    F.min("observation_time").alias("first_observation"),
    F.max("observation_time").alias("last_observation"),
)
calendar_years = spark.range(2017, 2025).select(F.col("id").cast("int").alias("year"))
calendar_months = spark.range(1, 13).select(F.col("id").cast("int").alias("month"))
station_calendar = clean.select("station_id").distinct().crossJoin(calendar_years).crossJoin(calendar_months)
months = (
    station_calendar.join(months_observed, ["station_id", "year", "month"], "left")
    .withColumn("expected_slots", F.dayofmonth(F.last_day(F.make_date("year", "month", F.lit(1)))) * 288)
    .fillna({"valid_slots": 0, "observed_days": 0})
    .withColumn("coverage", F.col("valid_slots") / F.col("expected_slots"))
)
years = months.groupBy("station_id", "year").agg(
    F.sum("valid_slots").alias("valid_slots"),
    F.sum("expected_slots").alias("expected_slots"),
).withColumn("coverage", F.col("valid_slots") / F.col("expected_slots"))
days = clean.groupBy("station_id", "local_date").agg(F.count("*").alias("valid_slots"))
days = days.withColumn("complete_day", F.col("valid_slots") == 288)
spacing = Window.partitionBy("station_id").orderBy("epoch")
gaps = clean.withColumn("elapsed_seconds", F.col("epoch") - F.lag("epoch").over(spacing)).where("elapsed_seconds > 300").groupBy("year", "elapsed_seconds").count()
for name, frame in [("coverage_month", months), ("coverage_year", years), ("coverage_day", days), ("gaps", gaps)]:
    frame.write.mode("overwrite").parquet(f"{ROOT}/{name}")
    frame.show(10, truncate=False)

raw_count = sum(row["row_count"] for row in files)
excluded_count = rejected.count()
retained_count = clean.count()
display({"raw_rows": raw_count, "retained_rows": retained_count, "excluded_rows": excluded_count})
assert retained_count + excluded_count == raw_count


## Task 1 — patterns in the pre-test period

The 2017–2023 summaries are descriptive. Monthly and hourly means average eligible station groups equally; they do not treat absent readings as zero rainfall.

In [ ]:
pretest = clean.where("year <= 2023")
train_wet_p99 = pretest.where("year <= 2022 AND rain_mm > 0").agg(
    F.percentile_approx("rain_mm", 0.99, 10000).alias("p99")
).first()["p99"]

distribution_by_year = pretest.groupBy("year").agg(
    F.count("*").alias("valid_intervals"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.avg("rain_mm").alias("mean_mm_per_valid_interval"),
    F.avg(F.when(F.col("rain_mm") > 0, F.col("rain_mm"))).alias("conditional_wet_mean_mm"),
    F.percentile_approx(F.when(F.col("rain_mm") > 0, F.col("rain_mm")), [0.5, 0.9, 0.95, 0.99], 10000).alias("wet_quantiles_mm"),
    F.max("rain_mm").alias("max_observed_mm"),
).withColumn("training_wet_p99_mm", F.lit(float(train_wet_p99)))

station_month = pretest.groupBy("station_id", "year", "month").agg(
    F.sum("rain_mm").alias("observed_total_mm"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.avg(F.when(F.col("rain_mm") > 0, F.col("rain_mm"))).alias("wet_intensity_mm"),
    F.count("*").alias("valid_slots"),
).join(months.select("station_id", "year", "month", "coverage"), ["station_id", "year", "month"])
monthly = station_month.where(F.col("coverage") >= config["primary_coverage_cutoff"]).groupBy("year", "month").agg(
    F.avg("observed_total_mm").alias("mean_observed_total_mm"),
    F.avg("wet_frequency").alias("mean_wet_frequency"),
    F.avg("wet_intensity_mm").alias("mean_wet_intensity_mm"),
    F.count("*").alias("stations"),
)
distribution_by_year.orderBy("year").show(10, truncate=False)
monthly.orderBy("year", "month").show(84, truncate=False)


In [ ]:
station_annual = pretest.groupBy("station_id", "year").agg(
    F.sum("rain_mm").alias("observed_total_mm"),
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
).join(years.where("year <= 2023"), ["station_id", "year"])
annual_results = []
for cutoff in config["coverage_cutoffs"]:
    common = years.where("year <= 2023").groupBy("station_id").agg(
        F.min("coverage").alias("minimum_coverage"), F.countDistinct("year").alias("years")
    ).where((F.col("minimum_coverage") >= cutoff) & (F.col("years") == 7))
    annual_results.append(
        station_annual.join(common.select("station_id"), "station_id").groupBy("year").agg(
            F.avg("observed_total_mm").alias("mean_observed_total_mm"),
            F.avg("wet_frequency").alias("mean_wet_frequency"),
            F.count("*").alias("common_stations"),
        ).withColumn("coverage_cutoff", F.lit(cutoff))
    )
annual_common_cohort = reduce(lambda left, right: left.unionByName(right), annual_results)

station_hour = pretest.groupBy("station_id", "year", "month", "hour").agg(
    F.avg((F.col("rain_mm") > 0).cast("double")).alias("wet_frequency"),
    F.count("*").alias("valid_slots"),
).join(months.select("station_id", "year", "month", "coverage", "expected_slots"), ["station_id", "year", "month"])
station_hour = station_hour.where(F.col("coverage") >= config["primary_coverage_cutoff"]).withColumn(
    "hour_coverage", F.col("valid_slots") / (F.col("expected_slots") / 24)
).where(F.col("hour_coverage") >= config["primary_coverage_cutoff"])
hourly_by_year = station_hour.groupBy("year", "hour").agg(
    F.avg("wet_frequency").alias("equal_station_month_wet_frequency"),
    F.count("*").alias("station_month_groups"),
)
annual_common_cohort.orderBy("coverage_cutoff", "year").show(30, truncate=False)
hourly_by_year.orderBy("year", "hour").show(200, truncate=False)


In [ ]:
daily = pretest.groupBy("station_id", "year", "month", "local_date").agg(
    F.count("*").alias("valid_slots"), F.sum("rain_mm").alias("observed_daily_mm")
).withColumn("complete_day", F.col("valid_slots") == 288)
complete_daily = daily.where("complete_day").groupBy("year", "month").agg(
    F.avg("observed_daily_mm").alias("equal_station_mean_complete_day_mm"),
    F.count("*").alias("complete_station_days"),
)
calendar_days = months.where("year <= 2023").groupBy("year", "month").agg(
    F.sum(F.col("expected_slots") / 288).alias("fixed_network_calendar_days")
)
completeness = daily.groupBy("year", "month").agg(
    F.sum(F.col("complete_day").cast("long")).alias("complete_station_days")
).join(calendar_days, ["year", "month"], "right").fillna(0, ["complete_station_days"]).withColumn(
    "complete_fraction_of_calendar_days", F.col("complete_station_days") / F.col("fixed_network_calendar_days")
)

aggregates = {
    "distribution_by_year": distribution_by_year,
    "monthly": monthly,
    "annual_common_cohort": annual_common_cohort,
    "hourly_by_year": hourly_by_year,
    "complete_daily_patterns": complete_daily,
    "daily_completeness": completeness,
}
for name, frame in aggregates.items():
    frame.write.mode("overwrite").json(f"{ROOT}/task1_pretest/{name}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

figure_dir = PROJECT_ROOT / "figures" / "pretest"
figure_dir.mkdir(parents=True, exist_ok=True)
monthly_pd = monthly.orderBy("year", "month").toPandas()
hourly_pd = hourly_by_year.orderBy("year", "hour").toPandas()
annual_pd = annual_common_cohort.orderBy("coverage_cutoff", "year").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for year, group in monthly_pd.groupby("year"):
    axes[0].plot(group["month"], group["mean_observed_total_mm"], marker=".", label=str(year))
axes[0].set(xlabel="Month", ylabel="Mean observed station-month rainfall (mm)", xticks=range(1, 13))
axes[0].legend(fontsize=7, ncol=2)
for year, group in hourly_pd.groupby("year"):
    axes[1].plot(group["hour"], group["equal_station_month_wet_frequency"], label=str(year))
axes[1].set(xlabel="Hour of interval end (SGT)", ylabel="Equal station-month wet frequency", xticks=range(0, 24, 3))
axes[1].legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(figure_dir / "monthly_and_hourly_patterns.png", dpi=180)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
for cutoff, group in annual_pd.groupby("coverage_cutoff"):
    ax.plot(group["year"], group["mean_observed_total_mm"], marker="o", label=f"Coverage ≥ {cutoff:.0%}")
ax.set(xlabel="Year", ylabel="Mean observed annual station rainfall (mm)")
ax.legend()
fig.tight_layout()
fig.savefig(figure_dir / "annual_common_cohort.png", dpi=180)
plt.show()
plt.close(fig)
display({"train_wet_p99_mm": train_wet_p99, "figure_dir": str(figure_dir)})


## Read the results carefully

Wet frequency is calculated over valid five-minute observations. Monthly and hourly summaries are averages over eligible station groups. Annual comparisons require the same station in every analysed year at the selected coverage cutoff; if that cohort is empty, report the support limitation. Complete-day summaries use exactly 288 readings and may omit days with missing data.

In [ ]:
raw.unpersist()
clean.unpersist()
print("Cleaning and Task 1 summaries are saved under the selected run prefix.")
